# Week 5 — Gradio demo + HuggingFace Spaces deployment

Builds the same interface as `app.py` (the file your HuggingFace Space will
actually run) so what you test here in Colab is exactly what goes live —
no drive.mount() branch to remember to delete before deploying.


In [ ]:
# Run once per Colab session (safe to re-run: pulls latest instead of
# silently reusing a stale checkout from an earlier run this session).
import os
import subprocess
import sys

REPO_DIR = '/content/repo'
if not os.path.exists(REPO_DIR):
    # Replace with your own fork's URL if you're running this outside the
    # original solution repo.
    !git clone https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists -- pulling latest instead of re-cloning.')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)

!pip install -q --upgrade -r requirements.txt
# Force transformers (and only transformers -- not the whole requirements
# file, so this doesn't also re-download the much larger torch wheel) to
# the pinned exact version. A plain `-r requirements.txt` install can
# leave an already-installed version in place if pip's resolver treats it
# as already 'satisfying' things.
!pip install -q --upgrade --force-reinstall "transformers==4.46.3"

# Ground-truth check via subprocess (`pip show`), NOT `import transformers`
# -- if a cell anywhere earlier in this session already imported
# transformers, Python caches that in memory for the rest of the process,
# and reinstalling on disk doesn't change what's already loaded here. The
# actual training/eval scripts run as a separate `!python ...` subprocess
# each time, so they always see what's genuinely on disk -- this mirrors
# that instead of this notebook kernel's own (possibly stale) view.
result = subprocess.run(['pip', 'show', 'transformers'], capture_output=True, text=True)
installed_version = next(
    (l.split(':', 1)[1].strip() for l in result.stdout.splitlines() if l.startswith('Version:')),
    '?',
)
print('transformers installed on disk:', installed_version)

if 'src' not in sys.path:
    sys.path.insert(0, 'src')

if not installed_version.startswith('4.'):
    raise RuntimeError(
        f'transformers on disk is {installed_version}, not the pinned 4.46.3 -- '
        'pip could not satisfy the pin here, likely another package in this '
        'Colab image requires transformers>=5. Re-run this cell and share the '
        'full pip output above so the real conflicting package can be found.'
    )
print('Setup complete -- transformers', installed_version, 'is correctly installed on disk.')


## Step 1 — Try the interface in Colab

In [ ]:
import os
# Point at your saved model: a local Drive path while iterating in Colab,
# or your HuggingFace Hub repo id (e.g. 'yourusername/urdu-trocr-si26') once published.
os.environ['MODEL_PATH'] = '/content/drive/MyDrive/SI26-urdu-ocr-model/final'
os.environ['GRADIO_SHARE'] = 'true'

import importlib, app
importlib.reload(app)
app.interface.launch(share=True)


Open the printed `.gradio.live` link, upload one of your Urdu images, and **screenshot the working demo** — you'll need it for your README.

## Step 2 — Deploy on HuggingFace Spaces

1. Go to https://huggingface.co, click your profile icon, **New Space**.
2. Space name: `urdu-ocr-codesaviours-si26-[yourfirstname]`. SDK: **Gradio**. Visibility: **Public**.
3. Push this repo's `app.py`, `requirements.txt`, `src/`, `assets/`, and `examples/` to the Space
   (either `git push` the whole repo to the Space's git remote, or upload the files via the web UI).
4. In the Space's **Settings > Variables and secrets**, add a variable
   `MODEL_PATH` = your HuggingFace Hub model repo id from Week 4
   (e.g. `yourusername/urdu-trocr-si26`) — this is what makes the Space load
   your fine-tuned model instead of failing to find a local `models/` folder.
5. Commit. Wait 2-3 minutes, then visit your Space URL — your app should be live.

Your permanent Space URL:
`https://huggingface.co/spaces/[your-username]/urdu-ocr-codesaviours-si26-[yourfirstname]`

Copy this URL — you need it for your README's **Live demo** section.


## Step 3 — Write your README

Update the main `README.md` (already scaffolded with all 8 required sections —
see the repo root) with:
- your live demo link from Step 2,
- your CER/WER/accuracy numbers and dataset details from Weeks 3-4,
- a screenshot of the working demo,
- your name and credit line.

A strong README takes at least an hour. Don't rush it — it's the single most
visible part of your project.
